OBSLUGA — СПЕЦИФИКАЦИЯ БЭКЕНДА И ПРИВЯЗКИ ФРОНТЕНДА, v1
Документ самодостаточен: отдать агенту или исполнять как есть. Решения приняты, обсуждения не требуют.

[0] СТЕК И РАСКЛАДКА
- Python 3.10+, FastAPI, SQLAlchemy 2.0 (Mapped), Alembic (async), asyncpg, PostgreSQL, bcrypt, pyjwt, pydantic v2.
- Раскладка:
    app/main.py, app/config.py (.env: DATABASE_URL, JWT_SECRET, TOKEN_HOURS=12, TOKEN_HOURS_REMEMBER=720)
    app/database.py (async engine, sessionmaker, Base)
    app/models/{department,equipment,role,account,request}.py
    app/core/security.py (hash/verify, create/decode JWT), app/core/deps.py (get_current_account, require_perm)
    app/schemas/{auth,equipment,request}.py
    app/routers/{auth,equipment,tickets}.py
    app/seed.py (идемпотентный)
    static/ — текущий фронтенд без изменений структуры (index, login, admin, js/, styles.css)
- Фронтенд отдаёт то же приложение: StaticFiles(directory="static", html=True) монтируется на "/" ПОСЛЕ роутеров /api (CORS не нужен).

[1] МОДЕЛИ БД
Правила: строка back_populates равна имени атрибута на другой стороне; две FK из request на account разбираются foreign_keys=[колонка]; PK аккаунта account_id; импорт соседа: на детской стороне классом, на родительской строкой в аннотации + TYPE_CHECKING. department.number из модели и диаграммы исключён (решение).

department.py
    class Department(Base):
        __tablename__ = "department"
        department_id: Mapped[int] = mapped_column(primary_key=True)
        name: Mapped[str] = mapped_column(String(100), unique=True)
        equipment_items: Mapped[list["Equipment"]] = relationship(back_populates="department")
        accounts: Mapped[list["Account"]] = relationship(back_populates="department")

equipment.py
    class Equipment(Base):
        __tablename__ = "equipment"
        equipment_id: Mapped[int] = mapped_column(primary_key=True)
        department_id: Mapped[int] = mapped_column(ForeignKey("department.department_id"))
        name: Mapped[str] = mapped_column(String(255))
        department: Mapped[Department] = relationship(back_populates="equipment_items")
        requests: Mapped[list["Request"]] = relationship(back_populates="equipment")

role.py
    class Role(Base):
        __tablename__ = "role"
        role_id: Mapped[int] = mapped_column(primary_key=True)
        name: Mapped[str] = mapped_column(String(50), unique=True)
        perm_level: Mapped[int] = mapped_column(SmallInteger)   # 1 сотрудник, 2 инженер
        accounts: Mapped[list["Account"]] = relationship(back_populates="role")

account.py
    class Account(Base):
        __tablename__ = "account"
        account_id: Mapped[int] = mapped_column(primary_key=True)
        role_id: Mapped[int] = mapped_column(ForeignKey("role.role_id"))
        department_id: Mapped[int] = mapped_column(ForeignKey("department.department_id"))
        full_name: Mapped[str] = mapped_column(String(255))
        email: Mapped[str] = mapped_column(String(255), unique=True)
        password_hash: Mapped[str] = mapped_column(String(255))
        created_at: Mapped[datetime] = mapped_column(DateTime, server_default=func.now())
        updated_at: Mapped[Optional[datetime]] = mapped_column(DateTime, onupdate=func.now())
        role: Mapped[Role] = relationship(back_populates="accounts")
        department: Mapped[Department] = relationship(back_populates="accounts")

request.py
    class Request(Base):
        __tablename__ = "request"
        request_id: Mapped[int] = mapped_column(primary_key=True)
        equipment_id: Mapped[int] = mapped_column(ForeignKey("equipment.equipment_id"))
        reporter_id: Mapped[int] = mapped_column(ForeignKey("account.account_id"))
        engineer_id: Mapped[Optional[int]] = mapped_column(ForeignKey("account.account_id", ondelete="SET NULL"))
        malfunction: Mapped[str] = mapped_column(String(2048))          # в JS это description
        diagnosis: Mapped[Optional[str]] = mapped_column(String(2048))
        work_done: Mapped[Optional[str]] = mapped_column(String(2048))  # в JS это workDone, см [4]
        status: Mapped[str] = mapped_column(String(20), default="new")  # new | progress | closed
        created_at: Mapped[datetime] = mapped_column(DateTime, server_default=func.now())
        updated_at: Mapped[Optional[datetime]] = mapped_column(DateTime, onupdate=func.now())
        equipment: Mapped[Equipment] = relationship(back_populates="requests")
        reporter: Mapped[Account] = relationship(foreign_keys=[reporter_id])     # односторонняя
        engineer: Mapped[Optional[Account]] = relationship(foreign_keys=[engineer_id])  # односторонняя

Проверка: python -c "import app.models; from sqlalchemy.orm import configure_mappers; configure_mappers()"

[2] МИГРАЦИИ И СИД
- alembic init -t async; в env.py target_metadata = Base.metadata + импорт всех моделей; URL из app.config.
- Первая миграция autogenerate, ручная вычитка, alembic upgrade head.
- seed.py get-or-create по имени:
    roles: Сотрудник (perm 1), Инженер (perm 2)
    departments: Цех №1, Цех №2, Цех №3, Цех №4
    equipment (7) с привязкой к цеху: Токарный станок 16К20→Цех №1, Компрессор ВК-25→Цех №2, Конвейерная лента №3→Цех №3, Сварочный аппарат ТДМ-317→Цех №2, Насос ЦНС 60-99→Цех №4, Пресс гидравлический ПГ-100→Цех №1, Электропогрузчик EP-16→Цех №3
- registration: роль ищется по точному значению select («Сотрудник»/«Инженер»); цех — свободный текст, точное совпадение с именем department, иначе 400 со списком допустимых.

[3] БЕЗОПАСНОСТЬ
- security.py: hash_password = bcrypt.hashpw(pw.encode(), bcrypt.gensalt()).decode(); verify_password = bcrypt.checkpw; create_token: JWT HS256, payload { sub: account_id, perm: perm_level, dept: department_id, name: full_name, exp: now + (remember ? TOKEN_HOURS_REMEMBER : TOKEN_HOURS) }; decode_token -> payload или 401.
- deps.py: get_current_account из заголовка Authorization: Bearer (select joinedload role), 401 при отсутствии/битом токене; require_perm(level) — фабрика зависимостей, 403 при perm < level.

[4] СХЕМЫ (pydantic v2, model_config = ConfigDict(populate_by_name=True))
- auth: RegisterIn { full_name, email, department, role, password (min 8, хотя бы одна цифра) }; LoginIn { email, password, remember: bool = False }; ProfileOut { id, full_name, role, department }; TokenOut { token, profile: ProfileOut }.
- equipment: EquipmentOut { id, name }.
- request: RequestCreate { equipment_id: int, description: str 1..2048 }; RequestPatch { status: Optional[enum new|progress|closed] = None, diagnosis: Optional[str] = None, work_done: Optional[str] = Field(None, validation_alias=AliasChoices("workDone", "work_done"), serialization_alias="workDone") }; RequestOut { id, equipment: str, description: str, reporter: str, engineer: Optional[str], status, updated: str ISO, diagnosis: Optional[str], work_done с serialization_alias "workDone" }.
- RequestOut собирается в роутере вручную из джойнов: equipment.name, reporter.full_name, engineer.full_name или None, description = malfunction.

[5] РОУТЕРЫ (префикс /api, детали ошибок строками по-русски)
- POST /api/auth/register -> 201 TokenOut: department и role по точному имени (400 + список), email уникален (409), хеш пароля, аккаунт, токен с remember=False.
- POST /api/auth/login -> TokenOut: по email и verify_password, иначе 401 «Неверный email или пароль»; remember из тела.
- GET /api/auth/me -> ProfileOut текущего.
- GET /api/equipment -> EquipmentOut[], все, сортировка по name.
- GET /api/tickets -> RequestOut[]: joinedload equipment/reporter/engineer, сортировка updated desc nulls last, затем created desc; фильтры и поиск остаются на клиенте.
- POST /api/tickets (любой авторизованный) -> 201 RequestOut: reporter = текущий, engineer = NULL, status = new, malfunction = description, updated = now.
- PATCH /api/tickets/{id} (require_perm(2)) -> RequestOut: 404 при отсутствии; применить status/diagnosis/work_done; если engineer_id NULL и status != new, engineer_id = текущий; updated проставится onupdate.
- GET /api/health -> {"status": "ok"}.

[6] MAIN.PY
- include_router для auth/equipment/tickets с префиксом /api и тегами; health; ПОСЛЕДНИМ app.mount("/", StaticFiles(directory="static", html=True)).
- Запуск: uvicorn app.main:app --reload

[7] ПРИВЯЗКА ФРОНТЕНДА (точные замены, остальной JS не трогать)
- js/auth.js:
    login TODO -> fetch POST /api/auth/login { email, password, remember }; ok: remember ? localStorage.token : sessionStorage.token, туда же profile = JSON.stringify(profile), location = admin.html; ошибка -> showMessage(detail).
    register TODO -> fetch POST /api/auth/register { full_name, email, department, role, password }; ok: то же хранение токена, location = admin.html; ошибка -> showMessage(detail).
    В login.html проверить/добавить name у полей: email, password, remember, fullName, department, role, password, passwordConfirm.
- js/admin.js:
    const TOKEN = localStorage.token || sessionStorage.token; во все fetch заголовок Authorization: Bearer + TOKEN; ответ 401 -> location = login.html.
    DOMContentLoaded: fetch GET /api/tickets и GET /api/equipment -> массив заявок вместо MOCK_TICKETS; select оборудования заполнить option value = id, text = name; убрать MOCK_TICKETS целиком.
    renderTable: инженер -> escapeHtml(t.engineer || "—"); обновлено -> fmtDate(t.updated).
    Добавить fmtDate:
        function fmtDate(iso) {
          var d = new Date(iso);
          var months = ["янв","фев","мар","апр","мая","июн","июл","авг","сен","окт","ноя","дек"];
          return d.getDate() + " " + months[d.getMonth()] + ", " +
            String(d.getHours()).padStart(2,"0") + ":" + String(d.getMinutes()).padStart(2,"0");
        }
    create submit TODO -> fetch POST /api/tickets { equipment_id: Number(form.elements.equipment.value), description: form.elements.description.value }; ok: добавить ответ в массив, renderTable, toast «Заявка №id создана»; убрать nextId, department, reporter из логики; в admin.html убрать из createDialog поля «Цех / отдел» и «Сотрудник, подавший заявку» (оба значения берёт бэкенд из профиля).
    detail submit TODO -> fetch PATCH /api/tickets/{activeTicketId} { status, diagnosis, workDone }; ok: заменить заявку в массиве ответом, renderTable, toast; убрать мок-подстановку engineer = «Мария Кузнецова» (инженера ставит бэкенд).
    Сайдбар .admin-user-meta: fetch GET /api/auth/me -> strong = full_name, span = role + « · » + department; ссылка «Выйти»: click-обработчик чистит localStorage.token и sessionStorage.token, затем переход по href.
- js/app.js, index.html, styles.css: не трогать.

[8] РЕШЕНИЯ (фиксированы)
1. workDone: алиасы в схеме (AliasChoices + serialization_alias), JS без правок.
2. updated: ISO с бэкенда, формат «24 сен, 14:10» функцией fmtDate в admin.js.
3. engineer: null в API, «—» через || в renderTable.
4. department в TicketOut не отдаётся (текущий JS поле не использует); цех живёт в профиле account.
5. department.number исключён из модели и диаграммы.
6. Цех при регистрации: точное совпадение, иначе 400 со списком.
7. Фильтры и поиск: клиентские, как сейчас.
8. StaticFiles после роутеров, CORS не нужен.

[9] ПРОВЕРКА И РАНБУК
- python -c "import app.models; from sqlalchemy.orm import configure_mappers; configure_mappers()"
- alembic upgrade head; python -m app.seed; uvicorn app.main:app --reload
- Swagger: register сотрудник -> login -> POST ticket -> login инженер -> PATCH status=progress -> GET tickets: engineer = имя инженера, updated свежий.
- Фронт-смоук: login.html регистрация -> admin.html таблица из API -> создать заявку -> открыть -> сохранить результат -> бейдж и инженер обновились, тост без слова «демонстрационный».

[10] ЧЕКЛИСТ ФАЗ
0 модели + configure_mappers. 1 alembic + seed + сверка value select в login.html. 2 security + deps. 3 схемы. 4 роутеры + main + health. 5 привязка фронтенда по [7]. 6 тесты pytest + httpx (register/login/CRUD заявок/права perm), CI-workflow, README с инструкцией запуска, диаграмма с готовой базы для записки.